# ПЗ 8 — Постобработка результатов

**Задача:** читать результаты из Drive (ПЗ 3 и ПЗ 5), дедуплицировать текст и склеить детекции.

**Стек:** `pandas`, `rapidfuzz`

In [ ]:
!pip install rapidfuzz pandas -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

print('Файлы в результатах:')
for f in os.listdir(RESULTS_DIR):
    print(f'  {f}')

## Часть 1 — Дедупликация текста (OCR)

In [ ]:
import pandas as pd
from rapidfuzz import fuzz, process

OCR_CSV = f'{RESULTS_DIR}/ocr_results.csv'
df_ocr = pd.read_csv(OCR_CSV)
print(f'Строк до дедупликации: {len(df_ocr)}')

def deduplicate_texts(texts: list, threshold: int = 85) -> list:
    unique = []
    for t in texts:
        if not unique:
            unique.append(t)
            continue
        best = process.extractOne(t, unique, scorer=fuzz.ratio)
        if best is None or best[1] < threshold:
            unique.append(t)
    return unique

unique = deduplicate_texts(df_ocr['text'].dropna().tolist())
print(f'Строк после дедупликации: {len(unique)}')

df_dedup = pd.DataFrame({'text': unique})
OUT = f'{RESULTS_DIR}/ocr_dedup.csv'
df_dedup.to_csv(OUT, index=False, encoding='utf-8-sig')
print(f'Сохранено: {OUT}')
df_dedup.head(20)

## Часть 2 — Склейка детекций YOLO

In [ ]:
DET_CSV = f'{RESULTS_DIR}/yolo_detections.csv'
df_det = pd.read_csv(DET_CSV)

WINDOW = 5  # кадров — допустимый разрыв между срабатываниями одного класса

def merge_detections(group):
    group = group.sort_values('frame_num').reset_index(drop=True)
    events = []
    start = group.iloc[0]
    prev = start['frame_num']
    for _, row in group.iloc[1:].iterrows():
        if row['frame_num'] - prev > WINDOW:
            events.append({
                'class': start['class'],
                'start_frame': start['frame_num'],
                'end_frame': prev,
                'avg_conf': round(group['conf'].mean(), 3),
            })
            start = row
        prev = row['frame_num']
    events.append({
        'class': start['class'],
        'start_frame': start['frame_num'],
        'end_frame': prev,
        'avg_conf': round(group['conf'].mean(), 3),
    })
    return pd.DataFrame(events)

df_merged = df_det.groupby('class', group_keys=False).apply(merge_detections).reset_index(drop=True)
df_merged = df_merged.sort_values('start_frame').reset_index(drop=True)

OUT = f'{RESULTS_DIR}/yolo_merged.csv'
df_merged.to_csv(OUT, index=False)
print(f'Детекций: {len(df_det)} → Событий: {len(df_merged)}')
print(f'✅ Сохранено: {OUT}')
df_merged